# FinanceAI Agent Evaluation Dashboard

This notebook visualizes the metrics generated by `evaluation/run_evaluation.py`.

It automatically loads the latest evaluation CSV and summary JSON from:

```
evaluation/results/
```

and displays:
- Overall metrics
- Per-agent metrics
- Latency distribution
- Success rates
- Instruction adherence
- Error breakdown


In [1]:
from pathlib import Path
import pandas as pd
import json
import matplotlib.pyplot as plt

results_dir = Path("evaluation/results")

csv_file = max(results_dir.glob("evaluation_*.csv"), key=lambda p: p.stat().st_mtime)
summary_file = max(results_dir.glob("summary_*.json"), key=lambda p: p.stat().st_mtime)

print("CSV:", csv_file)
print("Summary:", summary_file)

df = pd.read_csv(csv_file)

with open(summary_file) as f:
    summary = json.load(f)

df.head()


ValueError: max() arg is an empty sequence

In [2]:
import pandas as pd

overall = pd.DataFrame([
    {
        "Metric":"Total Runs",
        "Value":summary["total_runs"]
    },
    {
        "Metric":"Completed Runs",
        "Value":summary.get("completed_runs",0)
    },
    {
        "Metric":"Success Rate (%)",
        "Value":summary["success_rate_pct"]
    },
    {
        "Metric":"Mean Latency (ms)",
        "Value":summary["mean_latency_ms"]
    },
    {
        "Metric":"P95 Latency (ms)",
        "Value":summary["p95_latency_ms"]
    },
    {
        "Metric":"Instruction Score",
        "Value":summary.get("mean_instruction_score",
                             summary.get("mean_instruction_adherence_score"))
    }
])

overall


NameError: name 'summary' is not defined

In [ ]:
cats = pd.DataFrame(summary["by_category"]).T
cats


In [ ]:
plt.figure(figsize=(8,4))
plt.bar(cats.index,cats["mean_latency_ms"])
plt.xticks(rotation=30)
plt.ylabel("Latency (ms)")
plt.title("Mean Latency by Agent")
plt.tight_layout()
plt.show()


In [ ]:
score_col="mean_instruction_score"
if score_col not in cats.columns:
    score_col="mean_adherence_score"

plt.figure(figsize=(8,4))
plt.bar(cats.index,cats[score_col])
plt.xticks(rotation=30)
plt.ylabel("Score")
plt.title("Instruction Adherence")
plt.tight_layout()
plt.show()


In [ ]:
if "status" in df.columns:
    df["status"].value_counts().plot(kind="bar",title="Run Status")
    plt.tight_layout()
    plt.show()


In [ ]:
plt.figure(figsize=(8,4))
df["latency_ms"].hist(bins=20)
plt.xlabel("Latency (ms)")
plt.ylabel("Count")
plt.title("Latency Distribution")
plt.tight_layout()
plt.show()


In [ ]:
display(df)

print("\nAverage response length:",df["response_words"].mean())

if "rating" in df.columns:
    print("\nRatings")
    display(df["rating"].value_counts(dropna=False))
